<a href="https://colab.research.google.com/github/Ersaoktaviannn/eeg-creative-state-classifier/blob/dev/EEG_Primary_Secondary.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **INSTALL / IMPORT LIBRARY**

In [ ]:
import os
import glob
import warnings
import numpy as np
import pandas as pd
import scipy.io as sio
import pywt

from scipy import signal
from scipy.stats import kruskal
from statsmodels.stats.multitest import multipletests

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import LeaveOneGroupOut, GroupKFold, GridSearchCV
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    classification_report,
    confusion_matrix,
)

warnings.filterwarnings("ignore")

# **CONFIG**

In [ ]:
# === Path data ===
# Sesuaikan dengan path Google Drive / local kamu
SECONDARY_DIR = "/content/drive/MyDrive/Creativity-Secondary"   # folder .mat data sekunder
PRIMARY_DIR = "/content/drive/MyDrive/Creativity-Primary/raw"   # folder .csv data primer
PRIMARY_METADATA_PATH = "/content/drive/MyDrive/Creativity-Primary/primary_metadata.csv"

# === Sampling ===
SFREQ_SECONDARY = 500
SFREQ_PRIMARY = 125
SFREQ_TARGET = 125

# === Channel yang digunakan ===
CHANNEL_NAMES = [
    "Fp1", "Fp2", "F3", "F4", "F7", "F8",
    "C3", "C4", "T3", "T4", "T5", "T6",
    "P3", "P4", "O1", "O2"
]

# Jika data sekunder kamu sudah benar urutannya sesuai 16 channel ini,
# maka index di bawah boleh tetap 0:16.
SELECTED_CHANNELS_IDX = list(range(16))

# === Label 5 kelas ===
LABEL_ORDER = ["RST1", "IDG", "IDE", "IDR", "RST2"]
LABEL_TO_ID = {label: i for i, label in enumerate(LABEL_ORDER)}
ID_TO_LABEL = {i: label for label, i in LABEL_TO_ID.items()}

# === Eksperimen epoch ===
EPOCH_LIST = [0.25, 0.5, 1.0]

# === Wavelet ===
WAVELET = "db4"

# Untuk epoch pendek, DWT level 1 lebih aman dan konsisten:
# 16 channel x 2 koefisien x 3 Hjorth = 96 fitur
DWT_MODE = "level1"

RANDOM_STATE = 42

# **BASIC SIGNAL PROCESSING**

In [ ]:
def resample_eeg(eeg, original_sfreq, target_sfreq=125):
    """
    eeg shape: channel x samples
    """
    if original_sfreq == target_sfreq:
        return eeg

    n_samples_target = int(eeg.shape[1] * target_sfreq / original_sfreq)
    eeg_resampled = signal.resample(eeg, n_samples_target, axis=1)
    return eeg_resampled


def notch_filter(eeg, sfreq, freq=50.0, quality=30):
    """
    Notch filter untuk mengurangi interferensi listrik 50 Hz.
    Jika sfreq 125 dan bandpass sampai 40 Hz, notch 50 Hz tetap aman sebagai step awal.
    """
    b, a = signal.iirnotch(w0=freq, Q=quality, fs=sfreq)
    return signal.filtfilt(b, a, eeg, axis=1)


def bandpass_filter(eeg, sfreq, low=1.0, high=40.0, order=4):
    """
    Bandpass 1-40 Hz sesuai rentang EEG kognitif utama.
    """
    nyq = sfreq / 2
    low_norm = low / nyq
    high_norm = high / nyq
    b, a = signal.butter(order, [low_norm, high_norm], btype="band")
    return signal.filtfilt(b, a, eeg, axis=1)


def average_reference(eeg):
    """
    Rereference ke average reference.
    """
    return eeg - np.mean(eeg, axis=0, keepdims=True)


def detect_bad_channels_by_variance(eeg, z_thresh=3.0):
    """
    Deteksi sederhana kanal ekstrem berdasarkan variansi.
    Output hanya index channel yang dicurigai buruk.
    Catatan: interpolasi spasial tidak dilakukan di fungsi ini.
    """
    ch_var = np.var(eeg, axis=1)
    std = np.std(ch_var)
    if std == 0:
        return []
    z = (ch_var - np.mean(ch_var)) / std
    bad_idx = np.where(np.abs(z) > z_thresh)[0].tolist()
    return bad_idx


def preprocess_eeg(eeg, original_sfreq, target_sfreq=125, apply_bad_channel_check=True):
    """
    Pipeline preprocessing yang sama untuk data primer dan sekunder.
    Urutan:
    1. Resampling ke 125 Hz
    2. Notch 50 Hz
    3. Bandpass 1-40 Hz
    4. Average reference
    5. Deteksi bad channel sederhana untuk logging
    """
    eeg = np.asarray(eeg, dtype=float)

    # Pastikan shape: channel x samples
    if eeg.shape[0] > eeg.shape[1]:
        eeg = eeg.T

    eeg = resample_eeg(eeg, original_sfreq, target_sfreq)
    eeg = notch_filter(eeg, target_sfreq, freq=50.0)
    eeg = bandpass_filter(eeg, target_sfreq, low=1.0, high=40.0)
    eeg = average_reference(eeg)

    bad_channels = []
    if apply_bad_channel_check:
        bad_idx = detect_bad_channels_by_variance(eeg)
        bad_channels = [CHANNEL_NAMES[i] for i in bad_idx if i < len(CHANNEL_NAMES)]

    return eeg, target_sfreq, bad_channels


# **HJORTH + DWT FEATURE EXTRACTION**

In [ ]:


def hjorth_parameters(x):
    """
    Menghitung Hjorth Activity, Mobility, Complexity.
    """
    x = np.asarray(x, dtype=float)

    if len(x) < 3:
        return [0.0, 0.0, 0.0]

    dx = np.diff(x)
    ddx = np.diff(dx)

    var_x = np.var(x)
    var_dx = np.var(dx)
    var_ddx = np.var(ddx)

    activity = var_x

    if var_x == 0:
        mobility = 0.0
    else:
        mobility = np.sqrt(var_dx / var_x)

    if var_dx == 0 or mobility == 0:
        complexity = 0.0
    else:
        mobility_dx = np.sqrt(var_ddx / var_dx)
        complexity = mobility_dx / mobility

    return [activity, mobility, complexity]


def extract_dwt_hjorth_features(epoch):
    """
    epoch shape: channel x samples
    Output: 96 fitur jika 16 channel.
    Untuk tiap channel:
    - DWT level 1 menghasilkan cA dan cD
    - masing-masing dihitung 3 Hjorth parameter
    Total: 16 x 2 x 3 = 96
    """
    features = []

    for ch in epoch:
        cA, cD = pywt.dwt(ch, WAVELET)
        features.extend(hjorth_parameters(cA))
        features.extend(hjorth_parameters(cD))

    return np.asarray(features, dtype=float)


def make_feature_names():
    names = []
    for ch in CHANNEL_NAMES:
        for coef in ["cA", "cD"]:
            for hp in ["activity", "mobility", "complexity"]:
                names.append(f"{ch}_{coef}_{hp}")
    return names

FEATURE_NAMES = make_feature_names()

# **SEGMENTATION**

In [ ]:


def segment_condition(eeg_segment, label, subject_id, source, sfreq, epoch_sec):
    """
    Membagi sinyal satu kondisi menjadi epoch tanpa overlap.
    """
    epoch_len = int(epoch_sec * sfreq)
    n_samples = eeg_segment.shape[1]
    n_epochs = n_samples // epoch_len

    X, y, groups, sources = [], [], [], []

    for i in range(n_epochs):
        start = i * epoch_len
        end = start + epoch_len
        epoch = eeg_segment[:, start:end]

        feat = extract_dwt_hjorth_features(epoch)

        X.append(feat)
        y.append(LABEL_TO_ID[label])
        groups.append(subject_id)
        sources.append(source)

    return X, y, groups, sources

# **LOAD SECONDARY DATA**

In [ ]:
def find_mat_key(mat_dict):
    """
    Mencari key data EEG dari file .mat.
    Mengabaikan key internal __header__, __version__, __globals__.
    """
    keys = [k for k in mat_dict.keys() if not k.startswith("__")]
    if len(keys) == 0:
        raise ValueError("Tidak ada key data valid di file .mat")

    # Ambil key dengan array numeric terbesar
    best_key = None
    best_size = -1
    for k in keys:
        val = mat_dict[k]
        if isinstance(val, np.ndarray) and val.size > best_size:
            best_key = k
            best_size = val.size

    return best_key


def load_secondary_file(mat_path):
    """
    Loader fleksibel untuk data sekunder .mat.
    Asumsi awal:
    - file .mat berisi array EEG
    - data diambil 16 channel yang sudah kamu validasi urutannya benar

    Jika struktur .mat berbeda, bagian ini yang perlu disesuaikan.
    """
    mat = sio.loadmat(mat_path)
    key = find_mat_key(mat)
    data = mat[key]

    data = np.asarray(data, dtype=float)

    # Pastikan shape channel x samples
    if data.shape[0] > data.shape[1]:
        data = data.T

    # Ambil 16 channel sesuai konfirmasi kamu
    eeg = data[SELECTED_CHANNELS_IDX, :]
    return eeg


def infer_label_from_filename(file_path):
    """
    Mengambil label dari nama file.
    Pastikan nama file mengandung salah satu label: RST1, IDG, IDE, IDR, RST2.
    """
    fname = os.path.basename(file_path).upper()
    for label in LABEL_ORDER:
        if label in fname:
            return label
    raise ValueError(f"Label tidak ditemukan pada nama file: {fname}")


def infer_subject_from_filename(file_path):
    """
    Mengambil subject ID dari nama file.
    Kamu boleh sesuaikan jika format nama file berbeda.
    Contoh aman: S001_RST1.mat -> S001
    """
    fname = os.path.basename(file_path)
    base = os.path.splitext(fname)[0]
    parts = base.replace("-", "_").split("_")

    # Ambil token pertama sebagai subject_id
    return parts[0]


def build_secondary_dataset(epoch_sec):
    """
    Build fitur dari seluruh file sekunder.
    """
    mat_files = sorted(glob.glob(os.path.join(SECONDARY_DIR, "*.mat")))

    X_all, y_all, groups_all, sources_all = [], [], [], []
    log_rows = []

    if len(mat_files) == 0:
        print("Tidak ada file sekunder ditemukan.")
        return np.empty((0, len(FEATURE_NAMES))), np.array([]), np.array([]), np.array([]), pd.DataFrame()

    for f in mat_files:
        try:
            subject_id = infer_subject_from_filename(f)
            label = infer_label_from_filename(f)

            eeg = load_secondary_file(f)
            eeg_clean, sfreq, bad_channels = preprocess_eeg(eeg, SFREQ_SECONDARY, SFREQ_TARGET)

            X, y, groups, sources = segment_condition(
                eeg_clean,
                label=label,
                subject_id=subject_id,
                source="secondary",
                sfreq=sfreq,
                epoch_sec=epoch_sec,
            )

            X_all.extend(X)
            y_all.extend(y)
            groups_all.extend(groups)
            sources_all.extend(sources)

            log_rows.append({
                "file": os.path.basename(f),
                "subject_id": subject_id,
                "label": label,
                "source": "secondary",
                "epoch_sec": epoch_sec,
                "n_epochs": len(X),
                "bad_channels": ",".join(bad_channels),
                "status": "OK",
            })

        except Exception as e:
            log_rows.append({
                "file": os.path.basename(f),
                "subject_id": None,
                "label": None,
                "source": "secondary",
                "epoch_sec": epoch_sec,
                "n_epochs": 0,
                "bad_channels": None,
                "status": f"ERROR: {e}",
            })

    return (
        np.asarray(X_all, dtype=float),
        np.asarray(y_all, dtype=int),
        np.asarray(groups_all),
        np.asarray(sources_all),
        pd.DataFrame(log_rows),
    )


# **LOAD PRIMARY DATA**

In [ ]:
def load_primary_csv(csv_path):
    """
    Loader data primer OpenBCI dalam format CSV.
    Wajib memiliki kolom channel:
    Fp1,Fp2,F3,F4,F7,F8,C3,C4,T3,T4,T5,T6,P3,P4,O1,O2
    """
    df = pd.read_csv(csv_path)

    missing = [c for c in CHANNEL_NAMES if c not in df.columns]
    if missing:
        raise ValueError(f"Kolom channel hilang di {csv_path}: {missing}")

    eeg = df[CHANNEL_NAMES].to_numpy(dtype=float).T
    return eeg


def build_primary_dataset(epoch_sec):
    """
    Build fitur dari data primer menggunakan metadata.

    Format primary_metadata.csv:
    subject_id,file_name,label,start_sec,end_sec
    P001,P001.csv,RST1,0,180
    P001,P001.csv,IDG,190,310
    P001,P001.csv,IDE,320,680
    P001,P001.csv,IDR,690,810
    P001,P001.csv,RST2,820,1000
    """
    if not os.path.exists(PRIMARY_METADATA_PATH):
        print("Metadata primer belum ditemukan. Primary dataset dilewati.")
        return np.empty((0, len(FEATURE_NAMES))), np.array([]), np.array([]), np.array([]), pd.DataFrame()

    meta = pd.read_csv(PRIMARY_METADATA_PATH)

    required_cols = ["subject_id", "file_name", "label", "start_sec", "end_sec"]
    missing_cols = [c for c in required_cols if c not in meta.columns]
    if missing_cols:
        raise ValueError(f"Kolom metadata primer kurang: {missing_cols}")

    X_all, y_all, groups_all, sources_all = [], [], [], []
    log_rows = []

    # Cache per file agar tidak baca CSV berulang
    eeg_cache = {}

    for _, row in meta.iterrows():
        try:
            subject_id = str(row["subject_id"])
            file_name = row["file_name"]
            label = str(row["label"]).upper()
            start_sec = float(row["start_sec"])
            end_sec = float(row["end_sec"])

            if label not in LABEL_TO_ID:
                raise ValueError(f"Label tidak valid: {label}")

            csv_path = os.path.join(PRIMARY_DIR, file_name)

            if csv_path not in eeg_cache:
                eeg_raw = load_primary_csv(csv_path)
                eeg_clean, sfreq, bad_channels = preprocess_eeg(eeg_raw, SFREQ_PRIMARY, SFREQ_TARGET)
                eeg_cache[csv_path] = (eeg_clean, sfreq, bad_channels)
            else:
                eeg_clean, sfreq, bad_channels = eeg_cache[csv_path]

            start_sample = int(start_sec * sfreq)
            end_sample = int(end_sec * sfreq)

            eeg_segment = eeg_clean[:, start_sample:end_sample]

            X, y, groups, sources = segment_condition(
                eeg_segment,
                label=label,
                subject_id=subject_id,
                source="primary",
                sfreq=sfreq,
                epoch_sec=epoch_sec,
            )

            X_all.extend(X)
            y_all.extend(y)
            groups_all.extend(groups)
            sources_all.extend(sources)

            log_rows.append({
                "file": file_name,
                "subject_id": subject_id,
                "label": label,
                "source": "primary",
                "epoch_sec": epoch_sec,
                "start_sec": start_sec,
                "end_sec": end_sec,
                "n_epochs": len(X),
                "bad_channels": ",".join(bad_channels),
                "status": "OK",
            })

        except Exception as e:
            log_rows.append({
                "file": row.get("file_name", None),
                "subject_id": row.get("subject_id", None),
                "label": row.get("label", None),
                "source": "primary",
                "epoch_sec": epoch_sec,
                "n_epochs": 0,
                "bad_channels": None,
                "status": f"ERROR: {e}",
            })

    return (
        np.asarray(X_all, dtype=float),
        np.asarray(y_all, dtype=int),
        np.asarray(groups_all),
        np.asarray(sources_all),
        pd.DataFrame(log_rows),
    )


# **BUILD COMBINED DATASET PER EPOCH**

In [ ]:
def build_dataset(epoch_sec, include_secondary=True, include_primary=True):
    datasets = []
    logs = []

    if include_secondary:
        Xs, ys, gs, ss, logs_sec = build_secondary_dataset(epoch_sec)
        datasets.append((Xs, ys, gs, ss))
        logs.append(logs_sec)

    if include_primary:
        Xp, yp, gp, sp, logs_pri = build_primary_dataset(epoch_sec)
        datasets.append((Xp, yp, gp, sp))
        logs.append(logs_pri)

    X_list, y_list, g_list, s_list = [], [], [], []

    for X, y, g, s in datasets:
        if len(X) > 0:
            X_list.append(X)
            y_list.append(y)
            g_list.append(g)
            s_list.append(s)

    if len(X_list) == 0:
        return (
            np.empty((0, len(FEATURE_NAMES))),
            np.array([]),
            np.array([]),
            np.array([]),
            pd.concat(logs, ignore_index=True) if logs else pd.DataFrame(),
        )

    X_all = np.vstack(X_list)
    y_all = np.concatenate(y_list)
    groups_all = np.concatenate(g_list)
    sources_all = np.concatenate(s_list)
    log_df = pd.concat(logs, ignore_index=True) if logs else pd.DataFrame()

    return X_all, y_all, groups_all, sources_all, log_df

# **MODELING FUNCTION: LOSO + GROUPKFOLD GRIDSEARCH**

In [ ]:


def get_knn_pipeline():
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("knn", KNeighborsClassifier())
    ])

    param_grid = {
        "knn__n_neighbors": [1, 3, 5, 7, 9, 11, 15, 21],
        "knn__weights": ["uniform", "distance"],
        "knn__metric": ["euclidean", "manhattan", "minkowski"],
    }

    return pipe, param_grid


def evaluate_loso(X, y, groups, scenario_name="LOSO"):
    """
    Evaluasi Leave-One-Subject-Out.
    GridSearch internal memakai GroupKFold agar tidak terjadi leakage antar subjek.
    """
    unique_groups = np.unique(groups)

    if len(unique_groups) < 3:
        raise ValueError("Minimal perlu 3 subject untuk GroupKFold internal.")

    logo = LeaveOneGroupOut()

    y_true_all = []
    y_pred_all = []
    fold_rows = []

    for fold, (train_idx, test_idx) in enumerate(logo.split(X, y, groups), start=1):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        groups_train = groups[train_idx]
        test_subject = np.unique(groups[test_idx])[0]

        # Inner CV berbasis subject
        n_inner = min(3, len(np.unique(groups_train)))
        inner_cv = GroupKFold(n_splits=n_inner)

        pipe, param_grid = get_knn_pipeline()

        grid = GridSearchCV(
            estimator=pipe,
            param_grid=param_grid,
            cv=inner_cv,
            scoring="f1_macro",
            n_jobs=-1,
            refit=True,
        )

        grid.fit(X_train, y_train, groups=groups_train)
        y_pred = grid.predict(X_test)

        y_true_all.extend(y_test)
        y_pred_all.extend(y_pred)

        fold_rows.append({
            "scenario": scenario_name,
            "fold": fold,
            "test_subject": test_subject,
            "n_test": len(y_test),
            "accuracy": accuracy_score(y_test, y_pred),
            "balanced_accuracy": balanced_accuracy_score(y_test, y_pred),
            "macro_f1": f1_score(y_test, y_pred, average="macro", zero_division=0),
            "weighted_f1": f1_score(y_test, y_pred, average="weighted", zero_division=0),
            "best_params": grid.best_params_,
        })

        print(f"[{scenario_name}] Fold {fold}/{len(unique_groups)} | Test={test_subject} | Macro F1={fold_rows[-1]['macro_f1']:.4f}")

    y_true_all = np.asarray(y_true_all)
    y_pred_all = np.asarray(y_pred_all)

    summary = {
        "scenario": scenario_name,
        "accuracy": accuracy_score(y_true_all, y_pred_all),
        "balanced_accuracy": balanced_accuracy_score(y_true_all, y_pred_all),
        "macro_precision": precision_score(y_true_all, y_pred_all, average="macro", zero_division=0),
        "macro_recall": recall_score(y_true_all, y_pred_all, average="macro", zero_division=0),
        "macro_f1": f1_score(y_true_all, y_pred_all, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true_all, y_pred_all, average="weighted", zero_division=0),
    }

    report = classification_report(
        y_true_all,
        y_pred_all,
        labels=list(range(len(LABEL_ORDER))),
        target_names=LABEL_ORDER,
        zero_division=0,
        output_dict=True,
    )

    cm = confusion_matrix(
        y_true_all,
        y_pred_all,
        labels=list(range(len(LABEL_ORDER)))
    )

    return {
        "summary": summary,
        "folds": pd.DataFrame(fold_rows),
        "report": pd.DataFrame(report).T,
        "confusion_matrix": pd.DataFrame(cm, index=LABEL_ORDER, columns=LABEL_ORDER),
        "y_true": y_true_all,
        "y_pred": y_pred_all,
    }


# **TRAIN SECONDARY, TEST PRIMARY**

In [ ]:
def evaluate_train_secondary_test_primary(X, y, groups, sources, scenario_name="secondary_to_primary"):
    train_mask = sources == "secondary"
    test_mask = sources == "primary"

    if train_mask.sum() == 0 or test_mask.sum() == 0:
        print(f"{scenario_name} dilewati karena data secondary/primary belum lengkap.")
        return None

    X_train, y_train, groups_train = X[train_mask], y[train_mask], groups[train_mask]
    X_test, y_test = X[test_mask], y[test_mask]

    n_inner = min(3, len(np.unique(groups_train)))
    inner_cv = GroupKFold(n_splits=n_inner)

    pipe, param_grid = get_knn_pipeline()

    grid = GridSearchCV(
        estimator=pipe,
        param_grid=param_grid,
        cv=inner_cv,
        scoring="f1_macro",
        n_jobs=-1,
        refit=True,
    )

    grid.fit(X_train, y_train, groups=groups_train)
    y_pred = grid.predict(X_test)

    summary = {
        "scenario": scenario_name,
        "accuracy": accuracy_score(y_test, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_test, y_pred),
        "macro_precision": precision_score(y_test, y_pred, average="macro", zero_division=0),
        "macro_recall": recall_score(y_test, y_pred, average="macro", zero_division=0),
        "macro_f1": f1_score(y_test, y_pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_test, y_pred, average="weighted", zero_division=0),
        "best_params": grid.best_params_,
    }

    report = classification_report(
        y_test,
        y_pred,
        labels=list(range(len(LABEL_ORDER))),
        target_names=LABEL_ORDER,
        zero_division=0,
        output_dict=True,
    )

    cm = confusion_matrix(y_test, y_pred, labels=list(range(len(LABEL_ORDER))))

    return {
        "summary": summary,
        "report": pd.DataFrame(report).T,
        "confusion_matrix": pd.DataFrame(cm, index=LABEL_ORDER, columns=LABEL_ORDER),
        "y_true": y_test,
        "y_pred": y_pred,
    }

# **STATISTICAL TEST: KRUSKAL-WALLIS 5 KELAS**

In [ ]:


def kruskal_feature_test(X, y):
    """
    Uji beda fitur antar 5 kelas menggunakan Kruskal-Wallis.
    Cocok jika distribusi fitur tidak normal.
    Koreksi multiple comparison menggunakan FDR Benjamini-Hochberg.
    """
    rows = []

    for i, feat_name in enumerate(FEATURE_NAMES):
        groups_per_label = []
        for label_id in range(len(LABEL_ORDER)):
            vals = X[y == label_id, i]
            if len(vals) > 0:
                groups_per_label.append(vals)

        if len(groups_per_label) < 2:
            stat, pval = np.nan, np.nan
        else:
            stat, pval = kruskal(*groups_per_label)

        rows.append({
            "feature": feat_name,
            "statistic": stat,
            "p_value": pval,
        })

    df = pd.DataFrame(rows)
    valid = df["p_value"].notna()

    corrected = np.full(len(df), np.nan)
    significant = np.full(len(df), False)

    if valid.sum() > 0:
        reject, p_corr, _, _ = multipletests(df.loc[valid, "p_value"], alpha=0.05, method="fdr_bh")
        corrected[valid.values] = p_corr
        significant[valid.values] = reject

    df["p_value_fdr"] = corrected
    df["significant_fdr_0_05"] = significant

    return df.sort_values("p_value_fdr")

# **RUN ALL EXPERIMENTS**

In [ ]:
all_summary_rows = []
all_logs = []
all_outputs = {}

for epoch_sec in EPOCH_LIST:
    print("\n" + "="*80)
    print(f"RUNNING EXPERIMENT | EPOCH = {epoch_sec} sec")
    print("="*80)

    X, y, groups, sources, log_df = build_dataset(epoch_sec, include_secondary=True, include_primary=True)
    log_df["epoch_sec"] = epoch_sec
    all_logs.append(log_df)

    print("Dataset shape:", X.shape)
    print("Jumlah subject:", len(np.unique(groups)) if len(groups) > 0 else 0)
    print("Distribusi source:")
    print(pd.Series(sources).value_counts() if len(sources) > 0 else "No data")
    print("Distribusi label:")
    print(pd.Series(y).map(ID_TO_LABEL).value_counts() if len(y) > 0 else "No data")

    if len(X) == 0:
        continue

    # -------------------------
    # Scenario A: Secondary only
    # -------------------------
    sec_mask = sources == "secondary"
    if sec_mask.sum() > 0 and len(np.unique(groups[sec_mask])) >= 3:
        result_sec = evaluate_loso(
            X[sec_mask], y[sec_mask], groups[sec_mask],
            scenario_name=f"secondary_loso_epoch_{epoch_sec}"
        )
        row = result_sec["summary"].copy()
        row["epoch_sec"] = epoch_sec
        row["dataset"] = "secondary"
        all_summary_rows.append(row)
        all_outputs[("secondary_loso", epoch_sec)] = result_sec

    # -------------------------
    # Scenario B: Primary only
    # -------------------------
    pri_mask = sources == "primary"
    if pri_mask.sum() > 0 and len(np.unique(groups[pri_mask])) >= 3:
        result_pri = evaluate_loso(
            X[pri_mask], y[pri_mask], groups[pri_mask],
            scenario_name=f"primary_loso_epoch_{epoch_sec}"
        )
        row = result_pri["summary"].copy()
        row["epoch_sec"] = epoch_sec
        row["dataset"] = "primary"
        all_summary_rows.append(row)
        all_outputs[("primary_loso", epoch_sec)] = result_pri

    # -------------------------
    # Scenario C: Train secondary, test primary
    # -------------------------
    result_cross = evaluate_train_secondary_test_primary(
        X, y, groups, sources,
        scenario_name=f"secondary_to_primary_epoch_{epoch_sec}"
    )
    if result_cross is not None:
        row = result_cross["summary"].copy()
        row["epoch_sec"] = epoch_sec
        row["dataset"] = "train_secondary_test_primary"
        all_summary_rows.append(row)
        all_outputs[("secondary_to_primary", epoch_sec)] = result_cross

    # -------------------------
    # Scenario D: Combined LOSO
    # -------------------------
    if len(np.unique(groups)) >= 3:
        result_combined = evaluate_loso(
            X, y, groups,
            scenario_name=f"combined_loso_epoch_{epoch_sec}"
        )
        row = result_combined["summary"].copy()
        row["epoch_sec"] = epoch_sec
        row["dataset"] = "combined"
        all_summary_rows.append(row)
        all_outputs[("combined_loso", epoch_sec)] = result_combined

    # -------------------------
    # Statistical test
    # -------------------------
    stat_df = kruskal_feature_test(X, y)
    all_outputs[("statistics", epoch_sec)] = stat_df


summary_df = pd.DataFrame(all_summary_rows)
logs_df = pd.concat(all_logs, ignore_index=True) if all_logs else pd.DataFrame()

print("\nSUMMARY RESULTS")
display(summary_df)

print("\nPROCESS LOG")
display(logs_df.head(20))

# **SHOW BEST RESULT**

In [ ]:
if len(summary_df) > 0:
    best_row = summary_df.sort_values("macro_f1", ascending=False).iloc[0]
    print("Best configuration based on Macro F1:")
    display(best_row.to_frame().T)

    best_epoch = best_row["epoch_sec"]
    best_dataset = best_row["dataset"]

    print(f"Best epoch: {best_epoch}")
    print(f"Best dataset scenario: {best_dataset}")
else:
    print("Belum ada hasil. Cek path data dan format file.")

# **DISPLAY CONFUSION MATRIX AND REPORT**

In [ ]:
# Contoh menampilkan hasil skenario tertentu:
# Ganti key sesuai kebutuhan.
# Pilihan key:
# ('secondary_loso', 0.25), ('secondary_loso', 0.5), ('secondary_loso', 1.0)
# ('primary_loso', 0.25), dst jika data primer tersedia
# ('combined_loso', 1.0), dst

selected_key = ("secondary_loso", 1.0)

if selected_key in all_outputs:
    print("Classification Report:")
    display(all_outputs[selected_key]["report"])

    print("Confusion Matrix:")
    display(all_outputs[selected_key]["confusion_matrix"])
else:
    print(f"Output {selected_key} belum tersedia.")

# **SAVE RESULTS**

In [ ]:
RESULT_DIR = "/content/drive/MyDrive/Creativity-Results"
os.makedirs(RESULT_DIR, exist_ok=True)

summary_path = os.path.join(RESULT_DIR, "summary_results.csv")
logs_path = os.path.join(RESULT_DIR, "processing_logs.csv")

summary_df.to_csv(summary_path, index=False)
logs_df.to_csv(logs_path, index=False)

for key, value in all_outputs.items():
    name, epoch_sec = key
    safe_epoch = str(epoch_sec).replace(".", "p")

    if isinstance(value, dict):
        if "report" in value:
            value["report"].to_csv(os.path.join(RESULT_DIR, f"report_{name}_{safe_epoch}.csv"))
        if "confusion_matrix" in value:
            value["confusion_matrix"].to_csv(os.path.join(RESULT_DIR, f"cm_{name}_{safe_epoch}.csv"))
        if "folds" in value:
            value["folds"].to_csv(os.path.join(RESULT_DIR, f"folds_{name}_{safe_epoch}.csv"), index=False)
    elif isinstance(value, pd.DataFrame):
        value.to_csv(os.path.join(RESULT_DIR, f"{name}_{safe_epoch}.csv"), index=False)

print("Saved to:")
print(summary_path)
print(logs_path)

# **TEMPLATE PRIMARY METADATA**

In [ ]:
primary_template = pd.DataFrame([
    {"subject_id": "P001", "file_name": "P001.csv", "label": "RST1", "start_sec": 0, "end_sec": 180},
    {"subject_id": "P001", "file_name": "P001.csv", "label": "IDG",  "start_sec": 190, "end_sec": 310},
    {"subject_id": "P001", "file_name": "P001.csv", "label": "IDE",  "start_sec": 320, "end_sec": 680},
    {"subject_id": "P001", "file_name": "P001.csv", "label": "IDR",  "start_sec": 690, "end_sec": 810},
    {"subject_id": "P001", "file_name": "P001.csv", "label": "RST2", "start_sec": 820, "end_sec": 1000},
])

print("Template primary_metadata.csv:")
display(primary_template)